# 에너지 수요 예측 데이터 전처리
**대회**: Enefit – Predict Energy Behavior of Prosumers 

---


## 목차
1. [경로 설정 및 로드](#load)
2. [데이터 전처리](#preprocessing)
   - 2.1 datetime 변환
   - 2.2 시간 파생 변수 생성
   - 2.3 필요한 컬럼 선택
3. [데이터 병합](#merge)
   - 3.1 Client 병합
   - 3.2 Weather 병합
   - 3.3 Electricity 병합
   - 3.4 Gas 병합
4. [특성 생성](#features)
   - 4.1 정렬 및 lag feature
5. [결측치 처리](#missing)
6. [시계열 Split](#split)
7. [CSV 저장](#save)

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

<a id="load"></a>
## 1️ 경로 설정 및 로드

프로젝트 경로를 설정하고 모든 필요한 데이터셋을 로드합니다.

In [13]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

# 파일명
TRAIN_PATH = DATA_DIR / "train.csv"
CLIENT_PATH = DATA_DIR / "client.csv"
ELECTRICITY_PATH = DATA_DIR / "electricity_prices.csv"
GAS_PATH = DATA_DIR / "gas_prices.csv"
HISTORICAL_WEATHER_PATH = DATA_DIR / "historical_weather.csv"

# 저장 경로
OUTPUT_DIR = DATA_DIR / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
train = pd.read_csv(TRAIN_PATH)
client = pd.read_csv(CLIENT_PATH)
electricity = pd.read_csv(ELECTRICITY_PATH)
gas = pd.read_csv(GAS_PATH)
historical_weather = pd.read_csv(HISTORICAL_WEATHER_PATH)

print("train:", train.shape)
print("client:", client.shape)
print("electricity:", electricity.shape)
print("gas:", gas.shape)
print("historical_weather:", historical_weather.shape)

train: (2018352, 9)
client: (41919, 7)
electricity: (15286, 4)
gas: (637, 5)
historical_weather: (1710802, 18)


<a id="preprocessing"></a>
## 2️ 데이터 전처리

날짜 변환, 파생 변수 생성, 필요한 컬럼을 선택합니다.

In [15]:
# =========================================================
# 2. datetime 변환
# =========================================================
train["datetime"] = pd.to_datetime(train["datetime"])
client["date"] = pd.to_datetime(client["date"])
electricity["forecast_date"] = pd.to_datetime(electricity["forecast_date"])
gas["forecast_date"] = pd.to_datetime(gas["forecast_date"])
historical_weather["datetime"] = pd.to_datetime(historical_weather["datetime"])

# merge 편하게 하려고 date 컬럼 통일
train["date"] = train["datetime"].dt.normalize()
client["date"] = client["date"].dt.normalize()
gas["date"] = gas["forecast_date"].dt.normalize()

print("\n[datetime 변환 완료]")
print(train[["datetime", "date"]].head())

# =========================================================
# 3. 시간 파생 변수 생성
# =========================================================
train["hour"] = train["datetime"].dt.hour
train["weekday"] = train["datetime"].dt.weekday
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["dayofyear"] = train["datetime"].dt.dayofyear
train["weekofyear"] = train["datetime"].dt.isocalendar().week.astype("int16")

print("\n[시간 파생 변수 생성 완료]")
print(train[["datetime", "hour", "weekday", "month", "day", "dayofyear", "weekofyear"]].head())



[datetime 변환 완료]
    datetime       date
0 2021-09-01 2021-09-01
1 2021-09-01 2021-09-01
2 2021-09-01 2021-09-01
3 2021-09-01 2021-09-01
4 2021-09-01 2021-09-01

[시간 파생 변수 생성 완료]
    datetime  hour  weekday  month  day  dayofyear  weekofyear
0 2021-09-01     0        2      9    1        244          35
1 2021-09-01     0        2      9    1        244          35
2 2021-09-01     0        2      9    1        244          35
3 2021-09-01     0        2      9    1        244          35
4 2021-09-01     0        2      9    1        244          35


In [16]:
# =========================================================
# 4. 필요한 컬럼만 선택
# =========================================================
train_cols = [
    "row_id",
    "prediction_unit_id",
    "county",
    "is_business",
    "product_type",
    "is_consumption",
    "target",
    "datetime",
    "date",
    "hour",
    "weekday",
    "month",
    "day",
    "dayofyear",
    "weekofyear",
    "data_block_id",
]
train = train[train_cols].copy()

client_cols = [
    "county",
    "is_business",
    "product_type",
    "date",
    "eic_count",
    "installed_capacity",
]
client = client[client_cols].copy()

historical_weather_cols = [
    "datetime",
    "temperature",
    "dewpoint",
    "rain",
    "snowfall",
    "surface_pressure",
    "cloudcover_total",
    "windspeed_10m",
    "shortwave_radiation",
]
historical_weather = historical_weather[historical_weather_cols].copy()

print("\n[필요한 컬럼 선택 완료]")

electricity = electricity[["forecast_date", "euros_per_mwh"]].copy()
gas = gas[["forecast_date", "date", "lowest_price_per_mwh", "highest_price_per_mwh"]].copy()


[필요한 컬럼 선택 완료]


<a id="merge"></a>
## 3️ 데이터 병합

Client, Weather, Electricity, Gas 테이블들을 datetime 기준으로 병합합니다.

In [17]:
# =========================================================
# 5. client merge
# =========================================================
train = train.merge(
    client,
    on=["county", "is_business", "product_type", "date"],
    how="left"
)

print("\n[client merge 완료]")
print("shape:", train.shape)

# =========================================================
# 6. weather merge
#    여러 weather station 평균값을 datetime 기준으로 집계
# =========================================================
weather_agg = historical_weather.groupby("datetime", as_index=False).mean(numeric_only=True)

train = train.merge(
    weather_agg,
    on="datetime",
    how="left"
)

print("\n[historical_weather merge 완료]")
print("shape:", train.shape)

# =========================================================
# 7. electricity merge
# =========================================================
electricity = electricity.rename(columns={"forecast_date": "datetime"})

train = train.merge(
    electricity,
    on="datetime",
    how="left"
)

print("\n[electricity merge 완료]")
print("shape:", train.shape)

# =========================================================
# 8. gas merge
#    gas는 날짜 단위 기준으로 merge
# =========================================================
train = train.merge(
    gas[["date", "lowest_price_per_mwh", "highest_price_per_mwh"]],
    on="date",
    how="left"
)

print("\n[gas merge 완료]")
print("shape:", train.shape)




[client merge 완료]
shape: (2018352, 18)

[historical_weather merge 완료]
shape: (2018352, 26)

[electricity merge 완료]
shape: (2018352, 27)

[gas merge 완료]
shape: (2018352, 29)


In [18]:
# =========================================================
# 9. lag feature 생성 전 정렬
# =========================================================
train = train.sort_values(
    ["prediction_unit_id", "is_consumption", "datetime"]
).reset_index(drop=True)

print("\n[정렬 완료]")
print(train[["prediction_unit_id", "is_consumption", "datetime"]].head())

# =========================================================
# 10. lag feature 생성
# =========================================================
group_cols = ["prediction_unit_id", "is_consumption"]

train["lag_1"] = (
    train.groupby(group_cols)["target"]
    .shift(1)
    .astype("float32")
)

train["lag_24"] = (
    train.groupby(group_cols)["target"]
    .shift(24)
    .astype("float32")
)

print("\n[lag feature 생성 완료]")
print(train[["target", "lag_1", "lag_24"]].head(30))



[정렬 완료]
   prediction_unit_id  is_consumption            datetime
0                   0               0 2021-09-01 00:00:00
1                   0               0 2021-09-01 01:00:00
2                   0               0 2021-09-01 02:00:00
3                   0               0 2021-09-01 03:00:00
4                   0               0 2021-09-01 04:00:00

[lag feature 생성 완료]
     target       lag_1  lag_24
0     0.713         NaN     NaN
1     1.132    0.713000     NaN
2     0.490    1.132000     NaN
3     0.496    0.490000     NaN
4     0.149    0.496000     NaN
5     0.298    0.149000     NaN
6     1.271    0.298000     NaN
7    22.122    1.271000     NaN
8    64.257   22.122000     NaN
9   170.312   64.257004     NaN
10  286.533  170.311996     NaN
11  391.054  286.532990     NaN
12  480.938  391.053986     NaN
13  422.591  480.937988     NaN
14  303.120  422.591003     NaN
15  208.335  303.119995     NaN
16  225.405  208.335007     NaN
17  136.324  225.404999     NaN
18   61.903  1

<a id="features"></a>
## 4️ 특성 생성

정렬과 lag feature를 생성합니다.

In [19]:
# =========================================================
# 11. 결측치 처리(평균/보간)
#   날씨 / 가격 --> 선형 보간
#   lag       --> 보간 + 그룹 평균
#   고객 정보   --> 그룹 평균
#   나머지      --> 전체 평균
# =========================================================

na_before = train.isna().sum().sort_values(ascending=False).head(20)
print("\n[결측치 상위 20개 - 처리 전]")
print(na_before)

# -------------------------------
# 1) 시간순 정렬
# -------------------------------
train = train.sort_values(["prediction_unit_id", "is_consumption", "datetime"])

# -------------------------------
# 2) 컬럼 그룹 나누기
# -------------------------------
interp_cols = [
    "temperature",
    "dewpoint",
    "rain",
    "snowfall",
    "surface_pressure",
    "cloudcover_total",
    "windspeed_10m",
    "shortwave_radiation",
    "euros_per_mwh",
    "lowest_price_per_mwh",
    "highest_price_per_mwh",
]

lag_cols = ["lag_1", "lag_24"]

mean_cols = ["eic_count", "installed_capacity"]


# -------------------------------
# 3) 보간 (시계열 데이터)
# -------------------------------
for col in interp_cols:
    train[col] = (
        train.groupby(["prediction_unit_id", "is_consumption"])[col]
             .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )

# -------------------------------
# 4) lag 처리 (보간 + 평균)
# -------------------------------
for col in lag_cols:
    train[col] = (
        train.groupby(["prediction_unit_id", "is_consumption"])[col]
             .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )
    train[col] = (
        train.groupby(["prediction_unit_id", "is_consumption"])[col]
             .transform(lambda s: s.fillna(s.mean()))
    )

# -------------------------------
# 5) 정적 변수 평균 처리
# -------------------------------
for col in mean_cols:
    train[col] = (
        train.groupby(["county", "is_business", "product_type"])[col]
             .transform(lambda s: s.fillna(s.mean()))
    )

# -------------------------------
# 6) 마지막 남은 결측 처리 (전체 평균)
# -------------------------------
num_cols = train.select_dtypes(include="number").columns

train[num_cols] = train[num_cols].fillna(train[num_cols].mean())

# -------------------------------
# 결과 확인
# -------------------------------
na_after = train.isna().sum().sum()
print("\n[결측치 처리 완료]")
print("남은 결측치 개수:", na_after)


[결측치 상위 20개 - 처리 전]
eic_count                6240
installed_capacity       6240
surface_pressure         4810
dewpoint                 4810
temperature              4810
rain                     4810
snowfall                 4810
cloudcover_total         4810
shortwave_radiation      4810
windspeed_10m            4810
lag_24                   3840
euros_per_mwh            3386
highest_price_per_mwh    3120
lowest_price_per_mwh     3120
lag_1                     666
target                    528
dayofyear                   0
weekofyear                  0
month                       0
weekday                     0
dtype: int64

[결측치 처리 완료]
남은 결측치 개수: 0


<a id="missing"></a>
## 5️ 결측치 처리

보간, 평균 등 다양한 방식으로 결측치를 처리합니다.

In [20]:
# =========================================================
# 12. 시계열 기준 split
# =========================================================
split_date = train["datetime"].quantile(0.8)

train_processed = train[train["datetime"] < split_date].copy()
valid_processed = train[train["datetime"] >= split_date].copy()

print("\n[시계열 split 완료]")
print("split_date:", split_date)
print("train_processed:", train_processed.shape)
print("valid_processed:", valid_processed.shape)


[시계열 split 완료]
split_date: 2023-01-25 01:00:00
train_processed: (1614612, 31)
valid_processed: (403740, 31)


<a id="split"></a>
## 6️ 시계열 Split

시간순 기준으로 train/valid set을 분할합니다.

In [24]:
# =========================================================
# 13. CSV 저장
# =========================================================
full_path = OUTPUT_DIR / "full_processed.csv"
train_path = OUTPUT_DIR / "train_processed.csv"
valid_path = OUTPUT_DIR / "valid_processed.csv"

train.to_csv(full_path, index=False)
train_processed.to_csv(train_path, index=False)
valid_processed.to_csv(valid_path, index=False)

print("\n[저장 완료]")
print("full:", full_path)
print("train:", train_path)
print("valid:", valid_path)


[저장 완료]
full: c:\Users\AN\Desktop\kaggle\data\processed_data\full_processed.csv
train: c:\Users\AN\Desktop\kaggle\data\processed_data\train_processed.csv
valid: c:\Users\AN\Desktop\kaggle\data\processed_data\valid_processed.csv


<a id="save"></a>
## 7️ CSV 저장

전처리된 데이터를 CSV 파일로 저장합니다.

In [22]:
df = pd.read_csv(DATA_DIR / "processed_data" / "full_processed.csv")

print(df.columns.tolist())
df.head()

['row_id', 'prediction_unit_id', 'county', 'is_business', 'product_type', 'is_consumption', 'target', 'datetime', 'date', 'hour', 'weekday', 'month', 'day', 'dayofyear', 'weekofyear', 'data_block_id', 'eic_count', 'installed_capacity', 'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure', 'cloudcover_total', 'windspeed_10m', 'shortwave_radiation', 'euros_per_mwh', 'lowest_price_per_mwh', 'highest_price_per_mwh', 'lag_1', 'lag_24']


,row_id,prediction_unit_id,county,is_business,product_type,is_consumption,target,datetime,date,hour,...,snowfall,surface_pressure,cloudcover_total,windspeed_10m,shortwave_radiation,euros_per_mwh,lowest_price_per_mwh,highest_price_per_mwh,lag_1,lag_24
0,0,0,0,0,1,0,0.713,2021-09-01 00:00:00,2021-09-01,0,...,0.0,1009.517857,48.294643,5.114335,6.312500,92.51,45.23,46.32,0.713,0.713
1,122,0,0,0,1,0,1.132,2021-09-01 01:00:00,2021-09-01,1,...,0.0,1009.345536,42.803571,5.082341,3.892857,88.90,45.23,46.32,0.713,0.713
2,244,0,0,0,1,0,0.490,2021-09-01 02:00:00,2021-09-01,2,...,0.0,1008.944643,35.312500,5.035466,1.758929,87.35,45.23,46.32,1.132,0.713
3,366,0,0,0,1,0,0.496,2021-09-01 03:00:00,2021-09-01,3,...,0.0,1008.779464,33.491071,5.165675,0.330357,86.88,45.23,46.32,0.490,0.713
4,488,0,0,0,1,0,0.149,2021-09-01 04:00:00,2021-09-01,4,...,0.0,1008.670536,29.973214,5.284226,0.000000,88.43,45.23,46.32,0.496,0.713
